# Evaluate Individual Kraken Text Segments

This notebook uses the fine-tuned Kraken checkpoint on `.cursor/00000009.tif` and visualizes **only individual text segment polygons** from `segmentation.lines`.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path("/home/math/gupta/work/greek_byzantine/greek-foundation")
LOCAL_KRAKEN = REPO_ROOT / "kraken"
if str(LOCAL_KRAKEN) not in sys.path:
    sys.path.insert(0, str(LOCAL_KRAKEN))

IMAGE_PATH = REPO_ROOT / ".cursor" / "00000009.tif"
CHECKPOINT_PATH = REPO_ROOT / "outputs/kraken_region_finetune/runs/run-20260619_113721/best_region_polygon_0.2451.safetensors"

assert IMAGE_PATH.exists(), IMAGE_PATH
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH
IMAGE_PATH, CHECKPOINT_PATH

In [ ]:
from PIL import Image
from IPython.display import display

image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Image size: {image.size}")
display(image.copy())

In [ ]:
from kraken.configs import SegmentationInferenceConfig
from kraken.tasks import SegmentationTaskModel

segmentation_model = SegmentationTaskModel.load_model(CHECKPOINT_PATH)
segmentation = segmentation_model.predict(image, SegmentationInferenceConfig())

print("Detected individual text segments/lines:", len(segmentation.lines))

In [ ]:
from PIL import ImageDraw


def polygon(points):
    return [(int(x), int(y)) for x, y in points]


def draw_individual_text_segments(base_image, segmentation, *, label_first=50):
    overlay = Image.new("RGBA", base_image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    for idx, line in enumerate(segmentation.lines):
        if not getattr(line, "boundary", None):
            continue
        pts = polygon(line.boundary)
        if len(pts) < 3:
            continue

        # Individual text segment polygons only. No page-level regions are drawn here.
        draw.polygon(pts, fill=(0, 120, 255, 45))
        draw.line(pts + [pts[0]], fill=(0, 120, 255, 255), width=2)

        if idx < label_first:
            draw.text(pts[0], str(idx), fill=(0, 0, 255, 255))

    return Image.alpha_composite(base_image.convert("RGBA"), overlay).convert("RGB")


segment_overlay = draw_individual_text_segments(image, segmentation)
display(segment_overlay)

In [ ]:
OUTPUT_DIR = CHECKPOINT_PATH.parent / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

segments_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_individual_text_segments.png"
segment_overlay.save(segments_path)
print("Saved individual text segment overlay:", segments_path)